# D2_07 Train / Test / Deploy 나누기

> **오늘 질문:** 모델을 만들기 전에, 데이터를 어떻게 나눠야 '현장 성능'을 미리 알 수 있을까?

| 데이터 | 기간 | 쓰임 |
|---|---|---|
| Train | 과거 앞쪽 80% | 모델 학습 |
| Test | 과거 뒤쪽 20% | 마지막에 한 번 평가 |
| Deploy | 4월 1일 이후 | 배포 후 들어오는 새 데이터 (정답을 모른다고 가정) |

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비 📋

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_parquet("../../data/factory_table.parquet")   # 시간 순으로 정렬되어 있음
features = ["air_temp", "process_temp", "rpm", "torque", "tool_wear", "vibration"]

## 1. Deploy 데이터 먼저 떼어 두기 ✍️

4월 1일 이후는 '배포 후 새로 들어오는 데이터'로 따로 둡니다. 모델 개발에는 쓰지 않습니다.

> 변수 이름: `past`, `deploy`

In [ ]:
# 힌트: df[df["timestamp"] < "날짜"]

## 2. 시간 기준 분할 ✍️

과거 데이터의 앞 80%는 Train, 뒤 20%는 Test입니다. 섞지 않습니다.

> 변수 이름: `cut`, `train`, `test`

In [ ]:
# 힌트: 표.iloc[:cut], 표.iloc[cut:]

## 3. 무작위 분할과 비교 ✍️

`train_test_split`은 기본으로 데이터를 섞어서 나눕니다. Test의 기간을 확인해 보세요.

> 변수 이름: `train_r`, `test_r`

In [ ]:
# 힌트: train_test_split(표, test_size=0.2, random_state=42)

**Q1.** 무작위 분할의 Test 기간은 어떤가요? 고장 예측 모델을 이렇게 평가하면 무엇이 문제일까요?

> 답:


## 4. 고장 비율 비교 ✍️

분할마다 고장 비율이 얼마나 다른지 봅니다. `stratify`를 주면 Train·Test의 비율을 똑같이 맞춥니다.

In [ ]:
# 힌트: train_test_split(..., stratify=표["failure"])

**Q2.** 시간 분할에서는 Train과 Test의 고장 비율이 다릅니다. 이것은 잘못된 분할일까요?

> 답:


## 5. 전처리는 Train에서만 배운다 ✍️

스케일러의 기준(평균·표준편차)은 Train으로만 `fit`하고, Test·Deploy에는 `transform`만 합니다.

> 변수 이름: `scaler`

In [ ]:
# 힌트: 스케일러.fit(Train) → 스케일러.transform(Test)

In [ ]:
# 표준화 결과 확인: Train은 평균 0 근처, Test·Deploy는 0에서 벗어날 수 있음
check = pd.DataFrame({
    "Train": pd.DataFrame(X_train, columns=features).mean(),
    "Test": pd.DataFrame(X_test, columns=features).mean(),
    "Deploy": pd.DataFrame(X_deploy, columns=features).mean(),
})
check.round(2)

## 정리 💬

**Q3.** Test·Deploy의 표준화 평균이 0이 아닌 컬럼이 있습니다. 왜 그렇고, 전체 데이터로 `fit`했다면 무엇이 문제였을까요?

> 답:
